# VinTelligence DATATHON 2026 - Data QA & Integrity Check

Notebook này thực hiện kiểm tra tính toàn vẹn dữ liệu cho toàn bộ các bảng CSV trước khi qua *Phần 2: Trực quan hoá & Phân tích EDA* và *Phần 3: Mô hình Dự báo Doanh thu*.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [6]:
class DataIntegrityChecker:
    """Kiểm tra tính toàn vẹn dữ liệu cho bộ dữ liệu E-commerce."""

    DEFAULT_EXPECTED_FILES = [
        "products.csv",
        "customers.csv",
        "promotions.csv",
        "geography.csv",
        "orders.csv",
        "order_items.csv",
        "payments.csv",
        "shipments.csv",
        "returns.csv",
        "reviews.csv",
        "sales.csv",
        "sample_submission.csv",
        "inventory.csv",
        "inventory_enhanced.csv",
        "web_traffic.csv",
    ]

    ALLOWED_NULLABLE_COLUMNS = {
        "customers": {"gender", "age_group", "acquisition_channel"},
        "promotions": {"applicable_category", "promo_channel", "min_order_value"},
        "order_items": {"promo_id", "promo_id_2"},
    }

    VALID_SHIPMENT_STATUSES = {"shipped", "delivered", "returned"}

    def __init__(self, data_dir="../data", expected_files=None):
        self.data_dir = Path(data_dir)
        self.expected_files = expected_files or self.DEFAULT_EXPECTED_FILES
        self.tables = {}
        self.load_errors = {}
        self.load_summary = {}

    def _require_tables(self, table_names):
        missing = [name for name in table_names if name not in self.tables]
        if missing:
            raise KeyError(
                "Không tìm thấy bảng cần thiết để kiểm tra: " + ", ".join(sorted(missing))
            )

    @staticmethod
    def _print_test_result(test_name, violation_count):
        if violation_count == 0:
            print(f"- {test_name}: ✅ PASSED")
        else:
            print(f"- {test_name}: ❌ {violation_count} dòng vi phạm")

    def load_data(self):
        # Tự động quét toàn bộ file CSV trong thư mục data.
        csv_paths = sorted(self.data_dir.glob("*.csv"))
        available_files = {path.name for path in csv_paths}
        expected_files = set(self.expected_files)

        missing_expected_files = sorted(expected_files - available_files)
        unexpected_files = sorted(available_files - expected_files)

        for csv_path in csv_paths:
            table_name = csv_path.stem
            try:
                self.tables[table_name] = pd.read_csv(csv_path, low_memory=False)
            except Exception as exc:
                self.load_errors[table_name] = str(exc)

        self.load_summary = {
            "loaded_tables": sorted(self.tables.keys()),
            "missing_expected_files": missing_expected_files,
            "unexpected_files": unexpected_files,
            "load_errors": self.load_errors,
        }
        return self.tables

    def coerce_date_columns(self):
        conversion_rows = []

        for table_name, df in self.tables.items():
            # Ép tất cả cột có chứa từ khóa 'date' về datetime.
            date_cols = [col for col in df.columns if "date" in col.lower()]
            for col in date_cols:
                raw_non_null = int(df[col].notna().sum())
                coerced = pd.to_datetime(df[col], errors="coerce")
                parsed_non_null = int(coerced.notna().sum())
                parse_failures = max(raw_non_null - parsed_non_null, 0)

                self.tables[table_name][col] = coerced
                conversion_rows.append(
                    {
                        "table": table_name,
                        "column": col,
                        "raw_non_null": raw_non_null,
                        "parsed_non_null": parsed_non_null,
                        "parse_failures": parse_failures,
                    }
                )

        return pd.DataFrame(conversion_rows)

    def check_missing_values(self):
        summary_rows = []

        for table_name, df in self.tables.items():
            # Các cột nằm trong danh sách allow-list sẽ được bỏ qua khi cảnh báo bắt buộc.
            allowed_nullable = self.ALLOWED_NULLABLE_COLUMNS.get(table_name, set())
            mandatory_cols = [col for col in df.columns if col not in allowed_nullable]

            total_null_count = int(df.isna().sum().sum())
            mandatory_null_count = (
                int(df[mandatory_cols].isna().sum().sum()) if mandatory_cols else 0
            )

            summary_rows.append(
                {
                    "table": table_name,
                    "total_null_count": total_null_count,
                    "mandatory_null_count": mandatory_null_count,
                }
            )

        return pd.DataFrame(summary_rows).sort_values("table").reset_index(drop=True)

    def check_business_constraints(self):
        self._require_tables(["products", "promotions", "orders", "shipments"])

        products = self.tables["products"]
        product_violation_count = int((products["cogs"] >= products["price"]).sum())

        promotions = self.tables["promotions"]
        percentage_mask = promotions["promo_type"].eq("percentage")
        out_of_range_mask = np.logical_or(
            promotions["discount_value"] < 0, promotions["discount_value"] > 100
        )
        promo_violation_count = int((percentage_mask & out_of_range_mask).sum())

        merged_shipments = self.tables["shipments"][["order_id"]].merge(
            self.tables["orders"][["order_id", "order_status"]],
            on="order_id",
            how="left",
        )
        invalid_shipment_status_count = int(
            (~merged_shipments["order_status"].isin(self.VALID_SHIPMENT_STATUSES)).sum()
        )

        return {
            "products_cogs_lt_price": product_violation_count,
            "promotions_percentage_discount_range": promo_violation_count,
            "shipments_valid_order_status": invalid_shipment_status_count,
        }

    def check_referential_integrity(self):
        self._require_tables(
            [
                "orders",
                "customers",
                "geography",
                "order_items",
                "products",
                "payments",
                "inventory",
            ]
        )

        def orphan_count(child_df, child_key, parent_df, parent_key):
            return int((~child_df[child_key].isin(parent_df[parent_key])).sum())

        orders = self.tables["orders"]
        customers = self.tables["customers"]
        geography = self.tables["geography"]
        order_items = self.tables["order_items"]
        products = self.tables["products"]
        payments = self.tables["payments"]
        inventory = self.tables["inventory"].copy()

        results = {
            "orders_customer_id_fk": orphan_count(
                orders, "customer_id", customers, "customer_id"
            ),
            "orders_zip_fk": orphan_count(orders, "zip", geography, "zip"),
            "customers_zip_fk": orphan_count(customers, "zip", geography, "zip"),
            "order_items_product_id_fk": orphan_count(
                order_items, "product_id", products, "product_id"
            ),
            "order_items_order_id_fk": orphan_count(
                order_items, "order_id", orders, "order_id"
            ),
        }

        duplicate_payment_order_ids = int(payments["order_id"].duplicated(keep=False).sum())
        results["payments_order_id_unique_1_to_1"] = duplicate_payment_order_ids

        inventory_orphan_products = orphan_count(inventory, "product_id", products, "product_id")
        results["inventory_product_id_fk"] = inventory_orphan_products

        if "snapshot_date" in inventory.columns:
            month_key = inventory["snapshot_date"].dt.to_period("M").astype(str)
            inventory_duplicate_rows = int(
                pd.DataFrame(
                    {
                        "product_id": inventory["product_id"],
                        "snapshot_month": month_key,
                    }
                ).duplicated(keep=False).sum()
            )
        else:
            inventory_duplicate_rows = int(
                inventory[["product_id"]].duplicated(keep=False).sum()
            )

        results["inventory_one_row_per_product_month"] = inventory_duplicate_rows
        return results

    def run_integrity_report(self):
        self.load_data()

        print("=" * 80)
        print("BÁO CÁO KIỂM TRA DATA INTEGRITY")
        print("=" * 80)

        print("\n[0] Kiểm tra nạp dữ liệu")
        print(f"- Số bảng nạp thành công: {len(self.load_summary['loaded_tables'])}")
        if self.load_summary["missing_expected_files"]:
            print(
                "- Các file kỳ vọng bị thiếu: "
                + ", ".join(self.load_summary["missing_expected_files"])
            )
        else:
            print("- Các file kỳ vọng: ✅ PASSED")

        if self.load_summary["unexpected_files"]:
            print(
                "- Các file ngoài danh sách kỳ vọng: "
                + ", ".join(self.load_summary["unexpected_files"])
            )
        else:
            print("- File ngoài danh sách kỳ vọng: ✅ PASSED")

        if self.load_summary["load_errors"]:
            print("- Lỗi khi nạp dữ liệu:")
            for table_name, error_msg in self.load_summary["load_errors"].items():
                print(f"  + {table_name}: {error_msg}")

        print("\n[1] Kiểm tra Data Types & Missing Values")
        date_report = self.coerce_date_columns()
        if date_report.empty:
            print("- Ép kiểu cột date: ✅ PASSED (không tìm thấy cột chứa từ khóa date)")
        else:
            total_date_columns = int(date_report.shape[0])
            total_parse_failures = int(date_report["parse_failures"].sum())
            print(f"- Số cột date đã ép kiểu: {total_date_columns}")
            self._print_test_result("Parse date lỗi (về NaT)", total_parse_failures)

        missing_report = self.check_missing_values()
        for _, row in missing_report.iterrows():
            table = row["table"]
            total_null_count = int(row["total_null_count"])
            mandatory_null_count = int(row["mandatory_null_count"])

            if total_null_count == 0:
                print(f"- Null toàn bảng [{table}]: ✅ PASSED")
            else:
                print(f"- Null toàn bảng [{table}]: {total_null_count}")

            if mandatory_null_count > 0:
                print(
                    f"  ⚠️ CẢNH BÁO ĐẶC BIỆT: [{table}] có {mandatory_null_count} giá trị Null ở cột bắt buộc"
                )

        print("\n[2] Kiểm tra Ràng buộc Logic Kinh doanh")
        business_results = self.check_business_constraints()
        self._print_test_result("products.cogs < products.price", business_results["products_cogs_lt_price"])
        self._print_test_result(
            "promotions(promo_type=percentage): discount_value trong [0, 100]",
            business_results["promotions_percentage_discount_range"],
        )
        self._print_test_result(
            "shipments chỉ tham chiếu order_status hợp lệ",
            business_results["shipments_valid_order_status"],
        )

        print("\n[3] Kiểm tra Referential Integrity & Cardinality")
        ref_results = self.check_referential_integrity()
        self._print_test_result("orders.customer_id FK -> customers.customer_id", ref_results["orders_customer_id_fk"])
        self._print_test_result("orders.zip FK -> geography.zip", ref_results["orders_zip_fk"])
        self._print_test_result("customers.zip FK -> geography.zip", ref_results["customers_zip_fk"])
        self._print_test_result("order_items.product_id FK -> products.product_id", ref_results["order_items_product_id_fk"])
        self._print_test_result("order_items.order_id FK -> orders.order_id", ref_results["order_items_order_id_fk"])
        self._print_test_result("payments.order_id không được lặp (1:1 với orders)", ref_results["payments_order_id_unique_1_to_1"])
        self._print_test_result("inventory.product_id FK -> products.product_id", ref_results["inventory_product_id_fk"])
        self._print_test_result("inventory 1 dòng/sản phẩm/tháng", ref_results["inventory_one_row_per_product_month"])

        print("\nHoàn tất báo cáo Data Integrity.")


def run_integrity_report(data_dir="../data"):
    """Hàm wrapper để chạy toàn bộ báo cáo Data Integrity."""
    checker = DataIntegrityChecker(data_dir=data_dir)
    checker.run_integrity_report()
    return checker

In [7]:
checker = run_integrity_report(data_dir="../data")

BÁO CÁO KIỂM TRA DATA INTEGRITY

[0] Kiểm tra nạp dữ liệu
- Số bảng nạp thành công: 14
- Các file kỳ vọng bị thiếu: inventory_enhanced.csv
- File ngoài danh sách kỳ vọng: ✅ PASSED

[1] Kiểm tra Data Types & Missing Values
- Số cột date đã ép kiểu: 12
- Parse date lỗi (về NaT): ✅ PASSED
- Null toàn bảng [customers]: ✅ PASSED
- Null toàn bảng [geography]: ✅ PASSED
- Null toàn bảng [inventory]: ✅ PASSED
- Null toàn bảng [order_items]: 1152816
- Null toàn bảng [orders]: ✅ PASSED
- Null toàn bảng [payments]: ✅ PASSED
- Null toàn bảng [products]: ✅ PASSED
- Null toàn bảng [promotions]: 40
- Null toàn bảng [returns]: ✅ PASSED
- Null toàn bảng [reviews]: ✅ PASSED
- Null toàn bảng [sales]: ✅ PASSED
- Null toàn bảng [sample_submission]: ✅ PASSED
- Null toàn bảng [shipments]: ✅ PASSED
- Null toàn bảng [web_traffic]: ✅ PASSED

[2] Kiểm tra Ràng buộc Logic Kinh doanh
- products.cogs < products.price: ✅ PASSED
- promotions(promo_type=percentage): discount_value trong [0, 100]: ✅ PASSED
- shipments c